#  NQ Gaps — Optimización, Sizing y Validación Out-of-Sample
## Notebook A.03 del Hands-On: Masterclass de Diseño de Estrategias Cuantitativas

---

**Objetivo:** Explorar la superficie de optimización (meseta vs pico), comparar métodos de sizing, y ejecutar la validación final In-Sample vs Out-of-Sample.

> *"Si tu equity curve depende del valor exacto del parámetro, no tienes un edge: tienes un ajuste."*

### Conceptos del Masterclass que demostramos:
| Slide | Concepto |
|:---:|---|
| 10 | Optimización: Meseta vs Pico Aislado |
| 11 | Sizing: Fixed vs Reinversión vs Apalancamiento |
| 12 | Validación: In-Sample vs Out-of-Sample |
| 13 | Ficha final de la estrategia |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import sys; sys.path.insert(0, '.')
from nb_style import *

---
## 1. Carga de Datos y Reconstrucción del Dataset

In [ ]:
section_header('CARGA DE DATOS @NQ 5m → RTH', '')

df_5m = pd.read_csv('../@NQ_5m.csv', parse_dates=['TimeStamp'])
df_5m['TS_NY'] = pd.to_datetime(df_5m['TimeStamp']).dt.tz_convert('America/New_York')
df_5m['Date_NY'] = df_5m['TS_NY'].dt.date
df_5m['Hour_NY'] = df_5m['TS_NY'].dt.hour
df_5m['Min_NY']  = df_5m['TS_NY'].dt.minute

rth_mask = (
    ((df_5m['Hour_NY'] > 9) | ((df_5m['Hour_NY'] == 9) & (df_5m['Min_NY'] >= 30))) &
    ((df_5m['Hour_NY'] < 16) | ((df_5m['Hour_NY'] == 16) & (df_5m['Min_NY'] == 0)))
)
df_rth_5m = df_5m[rth_mask].copy()

rth_daily = df_rth_5m.groupby('Date_NY').agg(
    Date=('Date_NY', 'first'), Open=('Open', 'first'), High=('High', 'max'),
    Low=('Low', 'min'), Close=('Close', 'last'), Volume=('TotalVolume', 'sum'),
    Bars=('Close', 'count')
).reset_index(drop=True)
rth_daily = rth_daily[rth_daily['Bars'] >= 70].reset_index(drop=True)

rth_daily['prev_Close'] = rth_daily['Close'].shift(1)
rth_daily['Gap_Pts'] = rth_daily['Open'] - rth_daily['prev_Close']
tr = np.maximum(rth_daily['High'] - rth_daily['Low'],
     np.maximum((rth_daily['High'] - rth_daily['prev_Close']).abs(),
                (rth_daily['Low'] - rth_daily['prev_Close']).abs()))
rth_daily['ATR_14'] = tr.rolling(14).mean().shift(1)
rth_daily['Gap_ATR'] = rth_daily['Gap_Pts'] / rth_daily['ATR_14']
rth_daily['ATR_14_Pct'] = (rth_daily['ATR_14'] / rth_daily['prev_Close']) * 100
median_vol = rth_daily['ATR_14_Pct'].median()
rth_daily['Vol_Regime'] = np.where(rth_daily['ATR_14_Pct'] > median_vol, 'Alta Vol', 'Baja Vol')

# ER y Hurst
change_10 = (rth_daily['Close'] - rth_daily['Close'].shift(10)).abs()
vol_10 = (rth_daily['Close'] - rth_daily['Close'].shift(1)).abs().rolling(10).sum()
rth_daily['ER_10'] = (change_10 / vol_10).shift(1)

rth_daily = rth_daily.dropna(subset=['ATR_14', 'ER_10']).reset_index(drop=True)
rth_daily['Year'] = pd.to_datetime(rth_daily['Date']).dt.year

# Dataset filtrado para Fade Gap
df_fade = rth_daily[
    (rth_daily['Gap_ATR'].abs() > 0.10) &
    (rth_daily['Vol_Regime'] == 'Baja Vol') &
    (rth_daily['ER_10'] < 0.40)
].copy()

section_header('DATASET LISTO', '')
print(f'Días totales: {len(rth_daily):,}')
print(f'Trades Fade Gap filtrados: {len(df_fade):,}')
print(f'Años: {rth_daily["Year"].min()} – {rth_daily["Year"].max()}')

---
## 2. Superficie de Optimización — Meseta vs Pico (Slide 10)

Barremos todas las combinaciones de **Stop Loss** (0.3 a 2.5 ATR) × **Take Profit** (0.3 a 3.5 ATR) y visualizamos el Sharpe Ratio resultante.

> La pregunta NO es "¿cuál es el mejor punto?" sino **"¿hay una región estable donde muchos puntos cercanos son buenos?"**

In [ ]:
# ═══ BARRIDO DE OPTIMIZACIÓN SL × TP ═══
section_header('BARRIDO DE OPTIMIZACIÓN SL × TP', '🔎')

sl_range = np.arange(0.3, 2.6, 0.1)
tp_range = np.arange(0.3, 3.6, 0.1)

def backtest_fast(df, sl_atr, tp_atr):
    rets = []
    for _, row in df.iterrows():
        atr, op, hi, lo, cl = row['ATR_14'], row['Open'], row['High'], row['Low'], row['Close']
        is_long = row['Gap_Pts'] < 0
        if is_long:
            hit_tp = hi >= op + tp_atr * atr
            hit_sl = lo <= op - sl_atr * atr
            if hit_tp and not hit_sl: r = tp_atr
            elif hit_sl and not hit_tp: r = -sl_atr
            elif hit_tp and hit_sl: r = -sl_atr
            else: r = (cl - op) / atr
        else:
            hit_tp = lo <= op - tp_atr * atr
            hit_sl = hi >= op + sl_atr * atr
            if hit_tp and not hit_sl: r = tp_atr
            elif hit_sl and not hit_tp: r = -sl_atr
            elif hit_tp and hit_sl: r = -sl_atr
            else: r = (op - cl) / atr
        rets.append(r - COST_PCT/100)
    rets = np.array(rets)
    if rets.std() == 0: return 0, 0, 0
    sharpe = rets.mean() / rets.std() * np.sqrt(252)
    pf = rets[rets>0].sum() / abs(rets[rets<0].sum()) if (rets<0).any() else 0
    wr = (rets > 0).mean() * 100
    return sharpe, pf, wr

# Ejecutar barrido
results_grid = np.zeros((len(sl_range), len(tp_range)))
pf_grid = np.zeros_like(results_grid)

for i, sl in enumerate(sl_range):
    for j, tp in enumerate(tp_range):
        sharpe, pf, wr = backtest_fast(df_fade, sl, tp)
        results_grid[i, j] = sharpe
        pf_grid[i, j] = pf
    if i % 5 == 0:
        print(f'  SL = {sl:.1f} ATR completado ({i+1}/{len(sl_range)})')

print(f'\nBarrido completo: {len(sl_range)} × {len(tp_range)} = {len(sl_range)*len(tp_range)} combinaciones')

# ═══ HEATMAPS ═══
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Superficie de Optimización — Meseta vs Pico Aislado', 
             fontsize=18, fontweight='bold', color=COLORS['text_bright'], y=1.02)

# Panel A: Sharpe Ratio
ax = axes[0]
sl_labels = [f'{v:.1f}' for v in sl_range]
tp_labels = [f'{v:.1f}' for v in tp_range]
sns.heatmap(results_grid, ax=ax, cmap=CMAP_DIVERGENT, center=0,
            xticklabels=[l if i%3==0 else '' for i, l in enumerate(tp_labels)],
            yticklabels=[l if i%3==0 else '' for i, l in enumerate(sl_labels)],
            cbar_kws={'label': 'Sharpe Ratio'},
            linewidths=0.1, linecolor=COLORS['grid'])

# Marcar el máximo
max_idx = np.unravel_index(results_grid.argmax(), results_grid.shape)
max_sharpe = results_grid[max_idx]
ax.plot(max_idx[1]+0.5, max_idx[0]+0.5, 'o', color=COLORS['yellow'], markersize=12, 
        markeredgecolor='white', markeredgewidth=2)
ax.set_title(f'Sharpe Ratio\n(Máximo: {max_sharpe:.2f} en SL={sl_range[max_idx[0]]:.1f}/TP={tp_range[max_idx[1]]:.1f})',
             color=COLORS['text_bright'])
ax.set_xlabel('Take Profit (ATR)')
ax.set_ylabel('Stop Loss (ATR)')

# Panel B: Profit Factor
ax = axes[1]
sns.heatmap(pf_grid, ax=ax, cmap=CMAP_DIVERGENT, center=1.0,
            xticklabels=[l if i%3==0 else '' for i, l in enumerate(tp_labels)],
            yticklabels=[l if i%3==0 else '' for i, l in enumerate(sl_labels)],
            cbar_kws={'label': 'Profit Factor'},
            linewidths=0.1, linecolor=COLORS['grid'])
ax.set_title('Profit Factor', color=COLORS['text_bright'])
ax.set_xlabel('Take Profit (ATR)')
ax.set_ylabel('Stop Loss (ATR)')

plt.tight_layout()
plt.show()

# Identificar meseta
section_header('ANÁLISIS DE MESETA VS PICO', '')
good_mask = results_grid > (max_sharpe * 0.7)  # 70% del máximo
n_good = good_mask.sum()
total = results_grid.size
print(f'Sharpe máximo: {max_sharpe:.2f}')
print(f'Puntos con Sharpe > 70% del máximo: {n_good}/{total} ({n_good/total*100:.1f}%)')
if n_good / total > 0.10:
    print(' MESETA DETECTADA — La estrategia es robusta paraméticamente.')
else:
    print(' PICO AISLADO — Posible sobreajuste paramétrico.')

---
## 3. Sizing: Fixed vs Reinversión vs Kelly (Slide 11)

> *"El dimensionamiento de posición determina si tu edge te hace rico, te mantiene estable, o te lleva a la bancarrota."*

Tomamos la secuencia real de trades del Fade Gap y simulamos tres cuentas que operan exactamente los mismos trades pero con distinto sizing.

In [ ]:
# ═══ SIMULACIÓN DE 3 MÉTODOS DE SIZING ═══
section_header('SIMULACIÓN DE SIZING', '💰')

# Usar los parámetros del centro de la meseta
best_sl = sl_range[max_idx[0]]
best_tp = tp_range[max_idx[1]]
print(f'Parámetros seleccionados: SL={best_sl:.1f} ATR, TP={best_tp:.1f} ATR')

# Generar secuencia de retornos en R (1R = 1 unidad de riesgo)
trade_rets_R = []
for _, row in df_fade.iterrows():
    atr, op, hi, lo, cl = row['ATR_14'], row['Open'], row['High'], row['Low'], row['Close']
    is_long = row['Gap_Pts'] < 0
    if is_long:
        hit_tp = hi >= op + best_tp * atr
        hit_sl = lo <= op - best_sl * atr
        if hit_tp and not hit_sl: r = best_tp
        elif hit_sl and not hit_tp: r = -best_sl
        elif hit_tp and hit_sl: r = -best_sl
        else: r = (cl - op) / atr
    else:
        hit_tp = lo <= op - best_tp * atr
        hit_sl = hi >= op + best_sl * atr
        if hit_tp and not hit_sl: r = best_tp
        elif hit_sl and not hit_tp: r = -best_sl
        elif hit_tp and hit_sl: r = -best_sl
        else: r = (op - cl) / atr
    trade_rets_R.append(r)

trade_rets_R = np.array(trade_rets_R)
print(f'Trades: {len(trade_rets_R)}')
print(f'E[R]: {trade_rets_R.mean():.3f}')
print(f'Total: {trade_rets_R.sum():.1f}R')

# ─── Simular 3 cuentas ───
init_capital = INIT_CASH
risk_pct = 0.01  # 1% del capital por trade

# 1. Fixed: riesgo fijo = 1% del capital INICIAL
eq_fixed = [init_capital]
for r in trade_rets_R:
    pnl = init_capital * risk_pct * r  # Siempre misma cantidad en $
    eq_fixed.append(eq_fixed[-1] + pnl)

# 2. Reinversión: riesgo = 1% del capital ACTUAL
eq_reinv = [init_capital]
for r in trade_rets_R:
    pnl = eq_reinv[-1] * risk_pct * r  # Crece con la cuenta
    eq_reinv.append(max(eq_reinv[-1] + pnl, 1))

# 3. Apalancado: riesgo = 5% del capital actual (5x más agresivo)
eq_kelly = [init_capital]
for r in trade_rets_R:
    pnl = eq_kelly[-1] * 0.05 * r
    eq_kelly.append(max(eq_kelly[-1] + pnl, 1))

# ═══ GRÁFICO ═══
fig, axes = plt.subplots(2, 1, figsize=(16, 12))
fig.suptitle('Tres Formas de Crecer: Mismo Edge, Distinto Sizing',
             fontsize=18, fontweight='bold', color=COLORS['text_bright'], y=1.02)

ax = axes[0]
x = range(len(eq_fixed))
ax.plot(x, eq_fixed, color=COLORS['blue'], lw=2, label=f'Fixed (1% inicial) → ${eq_fixed[-1]:,.0f}')
ax.plot(x, eq_reinv, color=COLORS['green'], lw=2, label=f'Reinversión (1% actual) → ${eq_reinv[-1]:,.0f}')
ax.plot(x, eq_kelly, color=COLORS['red'], lw=2, label=f'Apalancado (5% actual) → ${eq_kelly[-1]:,.0f}')
ax.axhline(init_capital, color=COLORS['text_dim'], linestyle='--', alpha=0.3)
ax.set_ylabel('Equity ($)')
ax.set_xlabel('Trade #')
ax.set_title('Curvas de Equity — 3 Métodos de Sizing', color=COLORS['text_bright'])
ax.legend(fontsize=11, loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))

def max_drawdown_series(equity):
    eq = np.array(equity)
    peak = np.maximum.accumulate(eq)
    dd = (eq - peak) / peak * 100
    return dd

ax = axes[1]
dd_fixed = max_drawdown_series(eq_fixed)
dd_reinv = max_drawdown_series(eq_reinv)
dd_kelly = max_drawdown_series(eq_kelly)

ax.fill_between(range(len(dd_fixed)), dd_fixed, 0, color=COLORS['blue'], alpha=0.3,
                label=f'Fixed (Max DD: {min(dd_fixed):.1f}%)')
ax.fill_between(range(len(dd_reinv)), dd_reinv, 0, color=COLORS['green'], alpha=0.3,
                label=f'Reinversión (Max DD: {min(dd_reinv):.1f}%)')
ax.fill_between(range(len(dd_kelly)), dd_kelly, 0, color=COLORS['red'], alpha=0.3,
                label=f'Apalancado (Max DD: {min(dd_kelly):.1f}%)')

ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Trade #')
ax.set_title('Drawdowns — El Precio del Apalancamiento', color=COLORS['text_bright'])
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
plt.show()

section_header('LECCIÓN DE SIZING', '')
print(f'Fixed (1%):       ${eq_fixed[-1]:>12,.0f}  |  Max DD: {min(dd_fixed):.1f}%')
print(f'Reinversión (1%): ${eq_reinv[-1]:>12,.0f}  |  Max DD: {min(dd_reinv):.1f}%')
print(f'Apalancado (5%):  ${eq_kelly[-1]:>12,.0f}  |  Max DD: {min(dd_kelly):.1f}%')
print()
print('El apalancamiento amplifica TANTO las ganancias como las pérdidas.')

---
## 4.  Validación In-Sample vs Out-of-Sample (Slide 12)

Esta es la **prueba de fuego** de todo investigador cuantitativo. Dividimos la data en:
- **In-Sample (2000 – 2018):** Donde diseñamos y optimizamos la estrategia
- **Out-of-Sample (2019 – 2026):** Datos que la estrategia **NUNCA ha visto**

> *"Si tus métricas colapsan fuera de la muestra, no tienes un edge: tienes un ajuste."*

In [ ]:
# ═══ SPLIT IN-SAMPLE / OUT-OF-SAMPLE ═══
section_header('SPLIT IN-SAMPLE / OUT-OF-SAMPLE', '')

IS_CUTOFF = 2019
df_IS = df_fade[df_fade['Year'] < IS_CUTOFF].copy()
df_OOS = df_fade[df_fade['Year'] >= IS_CUTOFF].copy()

print(f'IN-SAMPLE:      {len(df_IS):>5,} trades  ({df_IS["Year"].min()}-{df_IS["Year"].max()})')
print(f'OUT-OF-SAMPLE:   {len(df_OOS):>5,} trades  ({df_OOS["Year"].min()}-{df_OOS["Year"].max()})')
print(f'Ratio IS/OOS:    {len(df_IS)/len(df_OOS):.1f}x')

# ─── Re-optimizar SOLO en In-Sample ───
print('\nRe-optimizando solo en In-Sample...')
best_sharpe_is = -np.inf
best_params_is = (1.0, 1.0)

for sl in np.arange(0.5, 2.1, 0.1):
    for tp in np.arange(0.5, 3.1, 0.1):
        sh, _, _ = backtest_fast(df_IS, sl, tp)
        if sh > best_sharpe_is:
            best_sharpe_is = sh
            best_params_is = (round(sl, 1), round(tp, 1))

sl_is, tp_is = best_params_is
print(f'Mejor IS: SL={sl_is} ATR, TP={tp_is} ATR (Sharpe={best_sharpe_is:.2f})')

# ─── Aplicar parámetros IS al OOS ───
def full_metrics(df, sl, tp):
    rets = []
    for _, row in df.iterrows():
        atr, op, hi, lo, cl = row['ATR_14'], row['Open'], row['High'], row['Low'], row['Close']
        is_long = row['Gap_Pts'] < 0
        if is_long:
            hit_tp = hi >= op + tp * atr
            hit_sl = lo <= op - sl * atr
            if hit_tp and not hit_sl: r = tp
            elif hit_sl and not hit_tp: r = -sl
            elif hit_tp and hit_sl: r = -sl
            else: r = (cl - op) / atr
        else:
            hit_tp = lo <= op - tp * atr
            hit_sl = hi >= op + sl * atr
            if hit_tp and not hit_sl: r = tp
            elif hit_sl and not hit_tp: r = -sl
            elif hit_tp and hit_sl: r = -sl
            else: r = (op - cl) / atr
        rets.append(r - COST_PCT/100)
    
    rets = np.array(rets)
    if len(rets) == 0 or rets.std() == 0:
        return {'N': 0, 'Sharpe': 0, 'WR': 0, 'PF': 0, 'E_R': 0, 'MaxDD': 0, 'Total_R': 0, 'rets': rets}
    
    sharpe = rets.mean() / rets.std() * np.sqrt(252)
    wr = (rets > 0).mean() * 100
    pf = rets[rets>0].sum() / abs(rets[rets<0].sum()) if (rets<0).any() else np.inf
    cum = np.cumsum(rets)
    peak = np.maximum.accumulate(cum)
    dd = peak - cum
    max_dd = dd.max()
    
    return {'N': len(rets), 'Sharpe': sharpe, 'WR': wr, 'PF': pf, 
            'E_R': rets.mean(), 'MaxDD': max_dd, 'Total_R': cum[-1], 'rets': rets}

m_IS = full_metrics(df_IS, sl_is, tp_is)
m_OOS = full_metrics(df_OOS, sl_is, tp_is)

# ═══ TABLA DE DEGRADACIÓN ═══
print('\n' + '═' * 80)
print(f'{"Métrica":.<30s} {"In-Sample":>12s} {"Out-of-Sample":>15s} {"Degradación":>12s} {"Señal":>8s}')
print('─' * 80)

metrics_compare = [
    ('Sharpe Ratio', m_IS['Sharpe'], m_OOS['Sharpe'], 30),
    ('Win Rate (%)', m_IS['WR'], m_OOS['WR'], 15),
    ('Profit Factor', m_IS['PF'], m_OOS['PF'], 25),
    ('E[R] por trade', m_IS['E_R'], m_OOS['E_R'], 30),
    ('Max Drawdown (R)', m_IS['MaxDD'], m_OOS['MaxDD'], 50),
    ('Total (R)', m_IS['Total_R'], m_OOS['Total_R'], 50),
]

for name, is_v, oos_v, thresh in metrics_compare:
    if is_v != 0:
        deg = abs(is_v - oos_v) / abs(is_v) * 100
    else:
        deg = 0
    signal = '' if deg <= thresh else ('' if deg <= thresh*2 else '💀')
    print(f'{name:.<30s} {is_v:>12.3f} {oos_v:>15.3f} {deg:>10.1f}% {signal:>8s}')

print('─' * 80)

In [ ]:
# ═══ GRÁFICO: EQUITY CURVE IS + OOS ═══
fig, axes = plt.subplots(2, 1, figsize=(16, 12))
fig.suptitle('Validación In-Sample vs Out-of-Sample — La Prueba de Fuego', 
             fontsize=18, fontweight='bold', color=COLORS['text_bright'], y=1.02)

# Panel A: Equity curves separadas
ax = axes[0]
cum_IS = np.cumsum(m_IS['rets'])
cum_OOS = np.cumsum(m_OOS['rets'])

# Plotear IS
ax.plot(range(len(cum_IS)), cum_IS, color=COLORS['blue'], lw=2, label=f'In-Sample ({IS_CUTOFF-1})')
# Plotear OOS (continuando desde donde terminó IS)
offset = cum_IS[-1] if len(cum_IS) > 0 else 0
oos_x = range(len(cum_IS), len(cum_IS) + len(cum_OOS))
ax.plot(oos_x, cum_OOS + offset, color=COLORS['orange'], lw=2.5, label='Out-of-Sample (2019-2026)')

# Línea divisoria
ax.axvline(len(cum_IS), color=COLORS['yellow'], lw=2, linestyle='--', alpha=0.8)
ax.text(len(cum_IS) + 2, max(cum_IS)*0.9, f'← IS | OOS →\n{IS_CUTOFF}', 
        fontsize=12, color=COLORS['yellow'], fontweight='bold')

ax.axhline(0, color=COLORS['text_dim'], linestyle='--', alpha=0.3)
ax.set_xlabel('Trade #')
ax.set_ylabel('Equity Acumulada (R)')
ax.set_title('Equity Curve Continua: In-Sample → Out-of-Sample', color=COLORS['text_bright'])
ax.legend(fontsize=12)

# Panel B: Semáforo visual
ax = axes[1]
ax.axis('off')
ax.set_facecolor(COLORS['bg'])

metrics_visual = [
    ('Sharpe Ratio', m_IS['Sharpe'], m_OOS['Sharpe'], 30),
    ('Win Rate (%)', m_IS['WR'], m_OOS['WR'], 15),
    ('Profit Factor', m_IS['PF'], m_OOS['PF'], 25),
    ('Expectancy E[R]', m_IS['E_R'], m_OOS['E_R'], 30),
    ('Max Drawdown (R)', m_IS['MaxDD'], m_OOS['MaxDD'], 50),
]

# Headers
for x, txt in [(0.05, 'MÉTRICA'), (0.35, 'IN-SAMPLE'), (0.55, 'OUT-OF-SAMPLE'), (0.78, 'DEGRADACIÓN')]:
    ax.text(x, 0.92, txt, fontsize=11, color=COLORS['text_dim'], fontweight='bold',
            transform=ax.transAxes, ha='left' if x < 0.1 else 'center')

ax.axhline(0.88, color=COLORS['blue'], lw=2, alpha=0.5, xmin=0.03, xmax=0.97)

for i, (name, is_v, oos_v, thresh) in enumerate(metrics_visual):
    y = 0.78 - i * 0.15
    deg = abs(is_v - oos_v) / abs(is_v) * 100 if is_v != 0 else 0
    
    # Color por semáforo
    if deg <= thresh:
        color = COLORS['green']
        emoji = ''
    elif deg <= thresh * 2:
        color = COLORS['orange']
        emoji = ''
    else:
        color = COLORS['red']
        emoji = '💀'
    
    ax.text(0.05, y, name, fontsize=13, color=COLORS['text'], fontweight='bold', transform=ax.transAxes)
    ax.text(0.35, y, f'{is_v:.2f}', fontsize=14, color=COLORS['blue'], transform=ax.transAxes, ha='center')
    ax.text(0.55, y, f'{oos_v:.2f}', fontsize=14, color=color, fontweight='bold', transform=ax.transAxes, ha='center')
    ax.text(0.78, y, f'{deg:.0f}% {emoji}', fontsize=14, color=color, fontweight='bold', transform=ax.transAxes, ha='center')
    
    # Línea de separación
    if i < len(metrics_visual) - 1:
        ax.axhline(y - 0.07, color=COLORS['grid'], lw=0.5, alpha=0.5, xmin=0.03, xmax=0.97)

ax.set_title('Semáforo Cuantitativo de Validación', fontsize=15, fontweight='bold', 
             color=COLORS['text_bright'], pad=20)

plt.tight_layout()
plt.show()

---
## 5. Ficha Final de la Estrategia — NQ Gap Fade

In [ ]:
# ═══ FICHA FINAL DE ESTRATEGIA ═══
fig, ax = plt.subplots(figsize=(14, 10))
ax.axis('off')
fig.patch.set_facecolor(COLORS['bg'])

# Título
ax.text(0.5, 0.96, ' FICHA DE ESTRATEGIA: NQ GAP FADE', fontsize=20, fontweight='bold',
        color=COLORS['text_bright'], ha='center', transform=ax.transAxes)
ax.axhline(0.93, color=COLORS['blue'], lw=3, xmin=0.1, xmax=0.9)

# Sección 1: Hipótesis
y = 0.88
ax.text(0.05, y, 'HIPÓTESIS', fontsize=14, fontweight='bold', color=COLORS['blue'], transform=ax.transAxes)
ax.text(0.05, y-0.04, 'X: Gap significativo en apertura RTH', fontsize=11, color=COLORS['text'], transform=ax.transAxes)
ax.text(0.05, y-0.07, 'Y: El precio revierte hacia el cierre previo', fontsize=11, color=COLORS['text'], transform=ax.transAxes)
ax.text(0.05, y-0.10, 'Z: Market makers reequilibran inventarios overnight', fontsize=11, color=COLORS['text'], transform=ax.transAxes)

# Sección 2: Parámetros
y = 0.72
ax.text(0.05, y, 'PARÁMETROS OPERATIVOS', fontsize=14, fontweight='bold', color=COLORS['green'], transform=ax.transAxes)
params = [
    f'Instrumento: @NQ (E-mini Nasdaq 100)',
    f'Timeframe: 5 minutos (RTH 09:30-16:00 NY)',
    f'Filtro Vol: Baja Volatilidad (ATR14% < mediana)',
    f'Filtro ER: Efficiency Ratio < 0.40',
    f'Gap mínimo: 0.10 ATR',
    f'Stop Loss: {sl_is:.1f} ATR (optimizado IS)',
    f'Take Profit: {tp_is:.1f} ATR (optimizado IS)',
]
for j, p in enumerate(params):
    ax.text(0.05, y - 0.035*(j+1), f'  • {p}', fontsize=10, color=COLORS['text'], transform=ax.transAxes)

# Sección 3: Métricas
y = 0.40
ax.text(0.05, y, 'MÉTRICAS DE VALIDACIÓN', fontsize=14, fontweight='bold', color=COLORS['orange'], transform=ax.transAxes)
ax.text(0.05, y-0.04, f'  In-Sample ({IS_CUTOFF-1}-):', fontsize=11, color=COLORS['blue'], fontweight='bold', transform=ax.transAxes)
ax.text(0.05, y-0.07, f'    N={m_IS["N"]}  Sharpe={m_IS["Sharpe"]:.2f}  WR={m_IS["WR"]:.1f}%  PF={m_IS["PF"]:.2f}', 
        fontsize=10, color=COLORS['text'], transform=ax.transAxes, family='monospace')
ax.text(0.05, y-0.11, f'  Out-of-Sample ({IS_CUTOFF}+):', fontsize=11, color=COLORS['orange'], fontweight='bold', transform=ax.transAxes)
ax.text(0.05, y-0.14, f'    N={m_OOS["N"]}  Sharpe={m_OOS["Sharpe"]:.2f}  WR={m_OOS["WR"]:.1f}%  PF={m_OOS["PF"]:.2f}', 
        fontsize=10, color=COLORS['text'], transform=ax.transAxes, family='monospace')

# Los 3 Mandamientos
y = 0.18
ax.axhline(y + 0.04, color=COLORS['purple'], lw=2, xmin=0.1, xmax=0.9)
ax.text(0.5, y, 'LOS 3 MANDAMIENTOS DEL TRADER CUANTITATIVO', fontsize=14, fontweight='bold',
        color=COLORS['purple'], ha='center', transform=ax.transAxes)
mandamientos = [
    '1. CAUSA ESTRUCTURAL — Nunca operes sin un "porque Z" validado antes de ver datos',
    '2. SALIDA CONSCIENTE — El win rate es vanidad; la expectancy es supervivencia',
    '3. MESETAS > PICOS — Diseña stops con MAE empírico y busca estabilidad paramétrica',
]
for j, m in enumerate(mandamientos):
    ax.text(0.10, y - 0.04*(j+1), m, fontsize=10, color=COLORS['text'], transform=ax.transAxes)

plt.tight_layout()
plt.show()

section_header('SERIE A COMPLETA — NQ GAP FADE', '🏁')
print(' Hipótesis formulada con causa estructural (Z)')
print(' Exploración y regímenes de volatilidad')
print(' MAE/MFE calculados y stops calibrados empíricamente')
print(' Superficie de optimización con meseta identificada')
print(' Sizing comparado (Fixed, Reinversión, Apalancamiento)')
print(' Validación In-Sample vs Out-of-Sample con semáforo')
print()
print('⏭️  Siguiente: Serie B — Gold (@GC) con Parabolic SAR')